In [1]:
import copy
import torch
import numpy as np
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from graph_datasets import GraphDataset
from graph_attention import CrossAttention
from performer_pytorch import NaiveAttention, FastAttention

/Users/aidandaly/miniforge3/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/aidandaly/miniforge3/lib/python3.10/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/Users/aidandaly/miniforge3/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


In [2]:
edge_index = torch.tensor([[0,1],
                           [1,0],
                           [1,2],
                           [2,1]], dtype=torch.long)

n_nodes = 3
n_tokens = 10
token_dim = 6
x = torch.tensor(np.random.uniform(0,1,size=(n_nodes, n_tokens, token_dim)), dtype=torch.float)
graph = Data(x=x, edge_index=edge_index.t().contiguous())

data = GraphDataset([graph])
loader = DataLoader(data, batch_size=1)

In [7]:
gca = CrossAttention(dim=token_dim, heads=2, dim_head=16, fast_attention=True)

In [ ]:
gca.fast_attention = NaiveAttenti

In [8]:
batch = next(iter(loader))
out, attn = gca(batch.x, batch.edge_index, output_attentions=True)

print(out.shape)
print(attn.shape)

out2 = gca(batch.x, batch.edge_index)

assert(torch.equal(out, out2))

out

torch.Size([4, 2, 10, 16]) torch.Size([4, 10, 10])
torch.Size([3, 10, 6])
torch.Size([3, 10, 10])


tensor([[[ 0.0900, -0.1772, -0.0878, -0.3327, -0.1279, -0.0820],
         [ 0.0903, -0.1771, -0.0872, -0.3335, -0.1299, -0.0812],
         [ 0.0903, -0.1777, -0.0880, -0.3322, -0.1283, -0.0812],
         [ 0.0899, -0.1769, -0.0872, -0.3332, -0.1284, -0.0830],
         [ 0.0908, -0.1778, -0.0873, -0.3330, -0.1301, -0.0804],
         [ 0.0899, -0.1773, -0.0880, -0.3329, -0.1286, -0.0812],
         [ 0.0916, -0.1775, -0.0867, -0.3327, -0.1306, -0.0802],
         [ 0.0909, -0.1763, -0.0861, -0.3332, -0.1293, -0.0824],
         [ 0.0904, -0.1769, -0.0871, -0.3331, -0.1286, -0.0821],
         [ 0.0908, -0.1778, -0.0876, -0.3323, -0.1290, -0.0809]],

        [[ 0.1182, -0.2041, -0.0723, -0.3722, -0.1236, -0.0877],
         [ 0.1181, -0.2014, -0.0705, -0.3729, -0.1239, -0.0915],
         [ 0.1189, -0.2007, -0.0695, -0.3747, -0.1267, -0.0905],
         [ 0.1187, -0.2025, -0.0714, -0.3742, -0.1248, -0.0925],
         [ 0.1186, -0.2003, -0.0693, -0.3745, -0.1266, -0.0899],
         [ 0.1189, -0.2

To verify that cross-attention is working:
- Three nodes configured in a line:  {0}--{1}--{2}
- Manually compute K, Q, V matrices for each node
- Assert:
  - H_0 = Attention(Q_0, K_1, V_1)
  - H_1 = (Attention(Q_1, K_0, V_0) + Attention(Q_1, K_2, V_2)) / 2
  - H_2 = Attention(Q_2, K_1, V_1)

In [72]:
n_nodes = 3
n_tokens = 4
token_dim = 2

edge_index = torch.tensor([[0,1],
                           [1,0],
                           [1,2],
                           [2,1]], dtype=torch.long)

x = torch.tensor([[[0,1],
                   [1,1],
                   [1,0],
                   [0,1]],
                  [[-1,-1],
                   [0,-2],
                   [-1,0],
                   [-.5,-.5]],
                  [[.4,.4],
                   [.2,.2],
                   [.1,.1],
                   [.1,.4]]])
graph = Data(x=x, edge_index=edge_index.t().contiguous())

data = GraphDataset([graph])
loader = DataLoader(data, batch_size=1)

In [73]:
gca = CrossAttention(dim=token_dim, heads=1, dim_head=2, fast_attention=False)
gca.eval()

CrossAttention()

In [74]:
batch = next(iter(loader))
out, attn_wts = gca(batch.x, batch.edge_index, output_attentions=True)
out

tensor([[[-0.2906, -0.4783],
         [-0.2809, -0.4778],
         [-0.2772, -0.4761],
         [-0.2906, -0.4783]],

        [[ 0.4957, -0.2632],
         [ 0.5016, -0.2617],
         [ 0.4885, -0.2658],
         [ 0.4912, -0.2651]],

        [[-0.2845, -0.4771],
         [-0.2859, -0.4769],
         [-0.2865, -0.4768],
         [-0.2875, -0.4773]]], grad_fn=<AddBackward0>)

In [75]:
attn_wts.shape

torch.Size([3, 4, 4])

In [76]:
attention = NaiveAttention()

q0, q1, q2 = gca.to_q(x[0]), gca.to_q(x[1]), gca.to_q(x[2])
k0, k1, k2 = gca.to_k(x[0]), gca.to_k(x[1]), gca.to_k(x[2])
v0, v1, v2 = gca.to_v(x[0]), gca.to_v(x[1]), gca.to_v(x[2])

a0 = attention(q0, k1, v1)
a1 = (attention(q1, k0, v0) + attention(q1, k2, v2)) / 2
a2 = attention(q2, k1, v1)

assert torch.equal(gca.to_out(a0), out[0])
assert torch.equal(gca.to_out(a1), out[1])
assert torch.equal(gca.to_out(a2), out[2])